# Array Traversal — Complete Pattern Guide

Array Traversal is **the foundation of every other DSA pattern**. Hash maps, two pointers, sliding window, prefix sums, binary search — they're all variations on "scan an array intelligently." If your traversal isn't automatic, every later pattern will feel harder than it is.

This lesson covers:

1. Why traversal is the bedrock pattern
2. The **8 core traversal patterns** you must own cold
3. The **4 in-place modification patterns**
4. Index arithmetic (boundaries, off-by-one, ranges)
5. Single-pass aggregation (min/max/sum/count without re-scans)
6. Conditional traversal — early exit, skip, accumulate-while
7. Multi-array / parallel traversal
8. Reverse and direction-aware traversal
9. Common interview applications (Best Time to Buy/Sell Stock, Maximum Subarray, Move Zeroes, Plus One, Rotate Array, etc.)
10. Edge cases & gotchas
11. Interview execution framework

By the end, you should be able to walk through any array problem in your head and write the right loop on the first try.

### Why this matters
- **Interviews:** ~30% of "easy" problems are pure traversal. Doing them in O(n) with one pass distinguishes a strong candidate.
- **Backend:** Iterating over query results, processing batches, validating payloads.
- **Data:** Every reduction, projection, and filter starts as a traversal.

---
## 1. The Mental Model

A traversal is a **single sweep of the array** where you maintain a small piece of state and update it at each step.

```python
state = initial_value
for x in arr:
    state = update(state, x)
return state
```

That's it. The "art" is choosing:
- **What state to maintain** (a number, an index, a pair, a dict, a flag).
- **How to update** it given the current element.
- **What to return** at the end.

Almost every "easy" array problem fits this template. Master it and you eliminate a whole layer of overthinking.

### The Two Loop Styles

You'll use both — know when to reach for each.

| Style | When |
|-------|------|
| `for x in arr:` | You only need the values |
| `for i, x in enumerate(arr):` | You need indices too (in-place writes, comparing neighbours, returning indices) |
| `for i in range(len(arr)):` | You need to control the index manually (skip, jump, modify) |

**Default to `enumerate`** — it's the cleanest way to get both.

---
## 2. Pattern 1 — Linear Scan (Find / Search)

Walk the array, return the first match (or `-1` / `None` if not found).

**Time:** O(n). **Space:** O(1).

In [ ]:
def linear_search(arr, target):
    for i, x in enumerate(arr):
        if x == target:
            return i
    return -1

print(linear_search([10, 20, 30, 40], 30))   # 2
print(linear_search([10, 20, 30, 40], 99))   # -1

**Variants:**
- First occurrence: return on first match (above).
- Last occurrence: scan in reverse, or remember the most recent match.
- Count occurrences: don't return early — keep counting.
- Find all indices: collect into a list (or use a comprehension).

In [ ]:
# All indices of target
def all_indices(arr, target):
    return [i for i, x in enumerate(arr) if x == target]

print(all_indices([1, 2, 3, 2, 4, 2], 2))   # [1, 3, 5]

---
## 3. Pattern 2 — Single-Pass Aggregation (min / max / sum / count)

Compute a summary in one sweep. The state is a single accumulator.

**Time:** O(n). **Space:** O(1).

In [ ]:
def find_min(arr):
    if not arr:
        return None              # ← handle empty input!
    best = arr[0]
    for x in arr[1:]:
        if x < best:
            best = x
    return best

print(find_min([4, 2, 7, 1, 5]))   # 1
print(find_min([]))                # None

def avg(arr):
    if not arr:
        return 0.0
    total = 0
    for x in arr:
        total += x
    return total / len(arr)

print(avg([1, 2, 3, 4, 5]))   # 3.0

**Built-ins shortcut:** `min(arr)`, `max(arr)`, `sum(arr)`, `len(arr)` are all O(n) (or O(1) for `len`). In interviews, you can use them when allowed — but **also know how to write the loop**, because interviewers often ask you to implement it manually.

### Tracking *both* the value and its index

In [ ]:
def argmax(arr):
    if not arr:
        return -1
    best_i = 0
    for i in range(1, len(arr)):
        if arr[i] > arr[best_i]:
            best_i = i
    return best_i

print(argmax([4, 2, 7, 1, 5]))   # 2 (the index of 7)

---
## 4. Pattern 3 — Running Aggregate / "Best So Far"

You maintain the **best result so far** while scanning. The trick is realising that for many problems, you only need to know one or two pieces of state from the past.

This is the engine behind:
- **Best Time to Buy and Sell Stock** — track min-so-far, update best profit
- **Maximum Subarray (Kadane's)** — track running sum, reset to 0 if negative
- **Maximum product / running max / running min**

In [ ]:
# Best Time to Buy and Sell Stock (LC 121) — O(n) time, O(1) space.
def max_profit(prices):
    if not prices:
        return 0
    min_price = prices[0]
    best = 0
    for p in prices[1:]:
        best = max(best, p - min_price)
        min_price = min(min_price, p)
    return best

print(max_profit([7, 1, 5, 3, 6, 4]))   # 5
print(max_profit([7, 6, 4, 3, 1]))      # 0

In [ ]:
# Kadane's Algorithm — Maximum Subarray (LC 53)
def max_subarray(arr):
    best_ending_here = arr[0]
    best_overall     = arr[0]
    for x in arr[1:]:
        best_ending_here = max(x, best_ending_here + x)
        best_overall     = max(best_overall, best_ending_here)
    return best_overall

print(max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4]))   # 6
print(max_subarray([-3, -1, -2]))                       # -1

**Internalise this template** — it shows up in 5–10 named interview problems:

```python
state = init_from(arr[0])
best  = state
for x in arr[1:]:
    state = update(state, x)
    best  = compare(best, state)
return best
```

---
## 5. Pattern 4 — Counting / Frequency

Single-pass count of occurrences. State is usually a dict or `Counter`.

In [ ]:
from collections import Counter

def freq(arr):
    counts = {}
    for x in arr:
        counts[x] = counts.get(x, 0) + 1
    return counts

print(freq("mississippi"))
print(Counter("mississippi"))

**Pattern variations:**
- Most common element: `Counter(arr).most_common(1)`
- First duplicate during the scan: track a `seen` set.
- Has any duplicate: `len(set(arr)) != len(arr)` — one-liner
- Majority element: Boyer-Moore voting (a clever single-pass algorithm)

In [ ]:
def has_dup(arr):
    return len(set(arr)) != len(arr)

def first_dup(arr):
    seen = set()
    for x in arr:
        if x in seen:
            return x
        seen.add(x)
    return None

print(first_dup([1, 2, 3, 2, 4]))   # 2
print(first_dup([1, 2, 3, 4]))      # None

---
## 6. Pattern 5 — Neighbour Comparison

Compare each element with its **neighbour(s)**. State is the previous element (or a count of streaks).

Used for:
- Is the array sorted?
- Longest increasing run / max consecutive ones
- Detect plateaus, peaks, valleys
- Remove adjacent duplicates

In [ ]:
def is_sorted(arr):
    for i in range(1, len(arr)):
        if arr[i] < arr[i-1]:
            return False
    return True

def is_sorted2(arr):
    return all(a <= b for a, b in zip(arr, arr[1:]))

print(is_sorted([1, 2, 2, 3]))   # True
print(is_sorted([1, 3, 2]))      # False

# Max Consecutive Ones (LC 485)
def max_consecutive_ones(nums):
    best = run = 0
    for x in nums:
        run = run + 1 if x == 1 else 0
        best = max(best, run)
    return best

print(max_consecutive_ones([1, 1, 0, 1, 1, 1]))   # 3

**Tip:** `zip(arr, arr[1:])` gives you `(prev, curr)` pairs in one Pythonic line — perfect for neighbour comparisons.

---
## 7. Pattern 6 — Early Exit / Short-Circuit

When you can decide the answer **before reaching the end**, return immediately. This converts worst-case O(n) into best-case O(1) on lucky inputs.

Built-ins `any()` and `all()` short-circuit automatically over a generator.

In [ ]:
def contains(arr, target):
    for x in arr:
        if x == target:
            return True
    return False

def contains2(arr, target):
    return any(x == target for x in arr)

def all_positive(nums):
    return all(n > 0 for n in nums)

print(contains([1, 2, 3, 4], 3))   # True
print(all_positive([1, 2, -1, 3])) # False

---
## 8. Pattern 7 — Build a Result Array

Walk the input and append/transform into a new array. The state is the output list itself.

Often a list comprehension is cleaner.

In [ ]:
def squares(arr):
    return [x * x for x in arr]

def even_squares(arr):
    return [x * x for x in arr if x % 2 == 0]

# Cumulative sums (prefix sums)
def prefix_sums(arr):
    out, total = [], 0
    for x in arr:
        total += x
        out.append(total)
    return out

print(squares([1, 2, 3]))           # [1, 4, 9]
print(even_squares([1, 2, 3, 4]))   # [4, 16]
print(prefix_sums([1, 2, 3, 4]))    # [1, 3, 6, 10]

**Prefix sums** unlock O(1) range-sum queries — they get their own DSA module later. The traversal pattern, though, is just "walk the array and accumulate."

---
## 9. Pattern 8 — Parallel / Multi-Array Traversal

Walk **two or more** arrays in lockstep. State per array if needed.

`zip` is the workhorse here.

In [ ]:
def add(a, b):
    return [x + y for x, y in zip(a, b)]

def num_correct(answers, expected):
    return sum(1 for a, e in zip(answers, expected) if a == e)

print(add([1, 2, 3], [10, 20, 30]))                      # [11, 22, 33]
print(num_correct(["A","B","C","A"], ["A","C","C","D"])) # 2

# Two-index merge of two sorted arrays
def merge_sorted(a, b):
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    out.extend(a[i:])
    out.extend(b[j:])
    return out

print(merge_sorted([1, 3, 5], [2, 4, 6]))

---
## 10. In-Place Modification — 4 Patterns

Many "easy" problems require modifying the array **in place** with **O(1) extra space**. These are popular interview territory.

### 10.1 Slow/Fast write pattern (overwrite)

`slow` = write index. `fast` = scan index. Copy items you want to keep to position `slow` and bump `slow`.

In [ ]:
# Remove Element (LC 27) — remove all occurrences of `val`, return new length
def remove_element(nums, val):
    slow = 0
    for fast in range(len(nums)):
        if nums[fast] != val:
            nums[slow] = nums[fast]
            slow += 1
    return slow

a = [3, 2, 2, 3]
n = remove_element(a, 3)
print(n, a[:n])   # 2 [2, 2]

# Remove Duplicates from Sorted Array (LC 26)
def remove_duplicates(nums):
    if not nums:
        return 0
    slow = 1
    for fast in range(1, len(nums)):
        if nums[fast] != nums[slow - 1]:
            nums[slow] = nums[fast]
            slow += 1
    return slow

a = [1, 1, 2, 2, 3]
n = remove_duplicates(a)
print(n, a[:n])   # 3 [1, 2, 3]

### 10.2 Move Zeroes / partition pattern

Same slow/fast idea — move all non-zero items left, then fill the tail with zeros.

In [ ]:
# Move Zeroes (LC 283) — keep relative order, do it in place.
def move_zeroes(nums):
    slow = 0
    for fast in range(len(nums)):
        if nums[fast] != 0:
            nums[slow], nums[fast] = nums[fast], nums[slow]
            slow += 1

a = [0, 1, 0, 3, 12]
move_zeroes(a)
print(a)   # [1, 3, 12, 0, 0]

### 10.3 Reverse in place (two pointers from ends)

In [ ]:
def reverse_in_place(arr):
    l, r = 0, len(arr) - 1
    while l < r:
        arr[l], arr[r] = arr[r], arr[l]
        l += 1
        r -= 1

a = [1, 2, 3, 4, 5]
reverse_in_place(a)
print(a)   # [5, 4, 3, 2, 1]

### 10.4 Rotate / shift

Rotating an array by k positions can be done with three reverses, in place, O(1) extra space — a classic interview trick.

In [ ]:
# Rotate Array (LC 189) — rotate right by k.
def rotate(nums, k):
    n = len(nums)
    if n == 0:
        return
    k %= n
    nums.reverse()
    nums[:k] = reversed(nums[:k])
    nums[k:] = reversed(nums[k:])

a = [1, 2, 3, 4, 5, 6, 7]
rotate(a, 3)
print(a)   # [5, 6, 7, 1, 2, 3, 4]

---
## 11. Index Arithmetic — Boundaries & Off-by-One

Most array bugs are off-by-one errors. Memorise these.

| `range(...)` | Yields |
|--------------|--------|
| `range(n)` | `0 .. n-1` (n items) |
| `range(1, n)` | `1 .. n-1` (n-1 items) |
| `range(0, n, 2)` | every other index |
| `range(n-1, -1, -1)` | reverse: `n-1 .. 0` |
| `range(len(arr))` | every valid index |

### Pair-iteration shapes
| Goal | Loop |
|------|------|
| Each item | `for x in arr` |
| Each (index, item) | `for i, x in enumerate(arr)` |
| Adjacent pairs | `for a, b in zip(arr, arr[1:])` |
| Triangular pairs (i < j) | `for i in range(n): for j in range(i+1, n)` |
| All pairs | `for i in range(n): for j in range(n)` |
| Last-to-first | `for x in reversed(arr)` |

In [ ]:
arr = [10, 20, 30, 40, 50]

# Sum of (i * arr[i])
print(sum(i * x for i, x in enumerate(arr)))

# Differences between neighbours
print([b - a for a, b in zip(arr, arr[1:])])

# Reverse traversal
for i in range(len(arr) - 1, -1, -1):
    print(i, arr[i])

### Edge cases to always check
1. **Empty array** — most loops handle this naturally; aggregations like `min`/`max` need a guard.
2. **Single element** — sliding/neighbour patterns may break.
3. **All duplicates** — counting/dedupe edge case.
4. **Sorted vs unsorted assumption**.
5. **Negative numbers** — Kadane, max-product, etc.
6. **Indices out of bounds** — when accessing `arr[i+1]`, ensure `i+1 < len(arr)`.

---
## 12. Direction-Aware Traversal

Sometimes the **order matters** — front-to-back gives a different answer than back-to-front.

Examples:
- "Find the **last** occurrence" — scan from the right.
- **Plus One** — propagate carry from the rightmost digit.
- **Daily Temperatures** — scan right-to-left with a stack.
- **Trapping Rain Water** — two-pointer, but logically a left-and-right max scan.

In [ ]:
# Plus One (LC 66) — increment the big-int represented by `digits`.
def plus_one(digits):
    digits = digits[:]              # don't mutate input
    for i in range(len(digits) - 1, -1, -1):
        if digits[i] < 9:
            digits[i] += 1
            return digits
        digits[i] = 0               # carry
    return [1] + digits             # all 9s → grew by one digit

print(plus_one([1, 2, 3]))     # [1, 2, 4]
print(plus_one([1, 2, 9]))     # [1, 3, 0]
print(plus_one([9, 9, 9]))     # [1, 0, 0, 0]

---
## 13. Named Interview Problems Built on Pure Traversal

| # | Problem | Pattern | LC |
|---|---------|---------|----|
| 1 | Find the maximum / minimum | Aggregation | — |
| 2 | Contains Duplicate | Set during scan | 217 |
| 3 | Single Number | XOR accumulate | 136 |
| 4 | Best Time to Buy/Sell Stock I | Best-so-far | 121 |
| 5 | Maximum Subarray (Kadane) | Best-so-far | 53 |
| 6 | Maximum Consecutive Ones | Run length | 485 |
| 7 | Move Zeroes | Slow/Fast write | 283 |
| 8 | Remove Element | Slow/Fast write | 27 |
| 9 | Remove Duplicates from Sorted | Slow/Fast write | 26 |
| 10 | Plus One | Reverse traversal | 66 |
| 11 | Rotate Array | 3-reverse trick | 189 |
| 12 | Reverse String / Reverse Array | Two pointers | 344 |
| 13 | Running Sum of 1d Array | Prefix accumulate | 1480 |
| 14 | Pivot Index | Prefix + total | 724 |
| 15 | Majority Element | Boyer-Moore | 169 |
| 16 | Product of Array Except Self | Two passes | 238 |
| 17 | Find Max in Mountain Array | Direction-aware | 852 |
| 18 | Squares of a Sorted Array | Two pointers | 977 |

---
## 14. How to Spot This Pattern in an Interview

Reach for plain traversal when:
- The input is a single array (no sorting/grouping requirement).
- You can answer the question by maintaining **O(1) state** as you scan.
- The brute force is already O(n) — no need to optimise further.
- You're tracking "best so far," "first occurrence," "running sum," etc.

Reach for **a richer pattern** when:
- You need to find a pair/triplet → hash map or two pointers.
- You need range queries → prefix sums.
- You need a fixed window → sliding window.
- You need monotonic relationships → stack.

---
## 15. Interview Execution Framework

Use this for **every** array-traversal problem:

1. **Restate the problem** in your own words.
2. **Edge cases first** — empty array, single element, all duplicates, negatives.
3. **State the brute force** — usually nested loops, O(n²). State complexity out loud.
4. **Identify the state** you need to maintain to do it in one pass.
5. **Code the loop** — use `enumerate` for clarity, `zip` for parallel/neighbour.
6. **Walk through a small example** by hand, line by line.
7. **Discuss complexity:** time O(n), space O(1) (or whatever it is).
8. **Mention follow-ups:** what if the array streams (no random access)? What if it's sorted? What if you need k results?

If you can do this on autopilot, you'll never freeze on an array problem again.

---

## Summary

- Traversal = walk + maintain state. That's it.
- 8 core patterns: linear scan, aggregation, best-so-far, frequency, neighbour compare, early exit, build result, parallel.
- 4 in-place patterns: slow/fast write, partition, reverse, rotate.
- Always check empty / single / negative edge cases.
- `enumerate` for index+value, `zip(arr, arr[1:])` for neighbour pairs.
- The big-name problems (Buy/Sell Stock, Kadane, Move Zeroes, Plus One) are all 5–10 lines once you spot the state.

You're ready. Open `drills.ipynb`, then `practice.ipynb`.